In [1]:
# 1. Importação das bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid')

In [2]:
# Ajuste o caminho caso o notebook esteja em outra pasta
url = 'https://docs.google.com/spreadsheets/d/1Ou9RFvSBZHRtVLkEi45qsjfY8aB4dT1bYFz-cdeuwFw/export?format=xlsx'

df = pd.read_excel(url)
print(df.shape)
df.head()

(22, 12)


,Carimbo de data/hora,nome,area,gerente,empresa,CSAT,faturamento,colaborativo,impacto,início,fim estimado,situacao
0,2026-09-24 15:31:26,Projeto FOX,Dados,Gabriel Marques,FOX,NaN,0,0,0,2026-12-31,2026-12-31,Iniciado
1,2026-09-23 15:31:26,Projeto Plutão,Sites,Gabriel Marques,Plutão,NaN,33823,0,0,2026-08-17,2026-11-17,Iniciado
2,2026-09-19 15:31:26,Projeto Diavicon,Mobile,Ottávio,Diavicon,NaN,25000,0,1,2026-04-02,2026-10-29,Em andamento
3,2026-09-17 15:31:26,Projeto Arthur,Dados,Pedro de Pádua,Asimov,NaN,0,0,0,2026-06-27,2026-09-01,Concluído
4,2026-09-18 15:31:26,Projeto JMorais,Sites,Ana beatriz,JMorais,NaN,5500,0,0,2026-03-12,2026-05-07,Em andamento


In [3]:
mapa_colunas = {
    'Coluna 1': 'data_registro',                      # timestamp do formulário
    'Coluna 2': 'nome_projeto',
    'Coluna 3': 'area',                                # dados / mobile / desktop / sites
    'Coluna 4': 'responsavel',
    'Coluna 5': 'cliente',
    'Coluna 6': 'valor_total_projeto',                 # valor contratado do projeto
    'Coluna 7': 'valor_faturamento_colaborativo',      # faturamento sem o custo
    'Coluna 8': 'valor_custo',                         # custo do projeto para a ej
    'Coluna 9': 'projeto_de_impacto',                  # 0/1 -> binário para previsão futura
    'Coluna 10': 'data_inicio',
    'Coluna 11': 'data_fim',
    'Coluna 12': 'status',                             # concluído / em andamento / iniciado
}

df = df.rename(columns=mapa_colunas)
df.head()


,Carimbo de data/hora,nome,area,gerente,empresa,CSAT,faturamento,colaborativo,impacto,início,fim estimado,situacao
0,2026-09-24 15:31:26,Projeto FOX,Dados,Gabriel Marques,FOX,NaN,0,0,0,2026-12-31,2026-12-31,Iniciado
1,2026-09-23 15:31:26,Projeto Plutão,Sites,Gabriel Marques,Plutão,NaN,33823,0,0,2026-08-17,2026-11-17,Iniciado
2,2026-09-19 15:31:26,Projeto Diavicon,Mobile,Ottávio,Diavicon,NaN,25000,0,1,2026-04-02,2026-10-29,Em andamento
3,2026-09-17 15:31:26,Projeto Arthur,Dados,Pedro de Pádua,Asimov,NaN,0,0,0,2026-06-27,2026-09-01,Concluído
4,2026-09-18 15:31:26,Projeto JMorais,Sites,Ana beatriz,JMorais,NaN,5500,0,0,2026-03-12,2026-05-07,Em andamento


In [4]:
df['projeto_de_impacto'] = df['projeto_de_impacto'].astype(bool)
df['status'] = df['status'].str.strip()
df['area'] = df['area'].str.strip()

print('Linhas duplicadas:', df.duplicated().sum())
print()
print('Valores nulos por coluna:')
print(df.isna().sum())
print()
print('Categorias de área:', df['area'].unique())
print('Categorias de status:', df['status'].unique())

KeyError: 'projeto_de_impacto'

In [ ]:
df['ano_inicio'] = df['data_inicio'].dt.year
df['mes_inicio'] = df['data_inicio'].dt.month
df['ano_mes_inicio'] = df['data_inicio'].dt.to_period('M').astype(str)
df['duracao_dias'] = (df['data_fim'] - df['data_inicio']).dt.days

df[['nome_projeto', 'data_inicio', 'data_fim', 'duracao_dias', 'ano_mes_inicio']].head()


In [ ]:
META_ANUAL = 88000

faturamento_mensal = (
    df.groupby('ano_mes_inicio', as_index=False)['valor_total_projeto']
      .sum()
      .sort_values('ano_mes_inicio')
      .reset_index(drop=True)
)
faturamento_mensal['faturamento_acumulado'] = faturamento_mensal['valor_total_projeto'].cumsum()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(faturamento_mensal['ano_mes_inicio'], faturamento_mensal['valor_total_projeto'],
        marker='o', label='Faturamento no mês')
ax.plot(faturamento_mensal['ano_mes_inicio'], faturamento_mensal['faturamento_acumulado'],
        marker='o', label='Faturamento acumulado')
ax.axhline(META_ANUAL, color='red', linestyle='--', label=f'Meta anual (R$ {META_ANUAL:,.0f})')
ax.set_xticklabels(faturamento_mensal['ano_mes_inicio'], rotation=45, ha='right')
ax.set_ylabel('R$')
ax.set_title('Evolução do faturamento')
ax.legend()
plt.tight_layout()
plt.show()

faturamento_mensal


In [ ]:
cluster_base = df.groupby('ano_mes_inicio').agg(
    faturamento_total=('valor_total_projeto', 'sum'),
    faturamento_colaborativo=('valor_faturamento_colaborativo', 'sum'),
    qtd_projetos=('nome_projeto', 'count'),
    qtd_projetos_impacto=('projeto_de_impacto', 'sum'),
    qtd_projetos_concluidos=('status', lambda s: (s == 'Concluído').sum()),
).reset_index()

cluster_base['pct_projetos_impacto'] = cluster_base['qtd_projetos_impacto'] / cluster_base['qtd_projetos']
cluster_base


In [ ]:
colunas_numericas = [
    'faturamento_total',
    'faturamento_colaborativo',
    'qtd_projetos',
    'qtd_projetos_impacto',
    'pct_projetos_impacto',
]

scaler = StandardScaler()
cluster_base_scaled = cluster_base.copy()
cluster_base_scaled[colunas_numericas] = scaler.fit_transform(cluster_base[colunas_numericas])
cluster_base_scaled
